# PetroVision Multimodal — DINOv2

Este notebook reproduz o recorte DeepCarbonate, extrai embeddings congelados do DINOv2-small e executa análises intra e cross-domain entre PPL e XPL. Execute as células em ordem.

## 1. Confirmar a GPU

Antes de continuar, selecione **Ambiente de execução > Alterar tipo de ambiente de execução > GPU**.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU não encontrada. Ative uma GPU no ambiente de execução do Colab.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch inicial:", torch.__version__)

## 2. Baixar o código e instalar dependências

In [ ]:
from pathlib import Path
import os
import subprocess

repo_dir = Path("/content/PetroVision-Multimodal")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/RangelGS/PetroVision-Multimodal.git", str(repo_dir)], check=True)
os.chdir(repo_dir)
print("Pasta atual:", Path.cwd())

In [ ]:
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Dependências instaladas.")

## 3. Validar o código

In [ ]:
subprocess.run([sys.executable, "-m", "pytest"], check=True)

## 4. Reconstruir o subconjunto DeepCarbonate

A simulação verifica a seleção antes de baixar. O download real das 336 imagens pode levar de 15 a 25 minutos.

In [ ]:
subprocess.run([sys.executable, "scripts/download_subset.py", "--dry-run"], check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/download_subset.py", "--yes"], check=True)

## 5. Auditar divisões, recriar catálogo e controlar a qualidade

In [ ]:
subprocess.run([sys.executable, "scripts/audit_split_integrity.py"], check=True)
subprocess.run([sys.executable, "scripts/build_catalog.py"], check=True)
subprocess.run([sys.executable, "scripts/run_quality_control.py"], check=True)

In [ ]:
import pandas as pd

near_duplicates = pd.read_csv("results/dinov2/tables/near_duplicate_report.csv")
quality_summary = pd.read_csv("results/tables/quality_summary.csv")
pending_near_duplicates = int(near_duplicates["manual_decision"].eq("pending_review").sum())
print(f"Candidatos perceptuais: {len(near_duplicates)}; pendentes: {pending_near_duplicates}")
display(near_duplicates)
display(quality_summary)
assert pending_near_duplicates == 0, "Existem candidatos perceptuais sem revisão."
assert int(quality_summary["accepted_for_model"].sum()) == 336, "Nem todas as imagens foram aprovadas."

## 6. Extrair embeddings DINOv2

O checkpoint é baixado na primeira execução. Os pesos permanecem congelados e somente o token global `CLS` é salvo.

In [ ]:
subprocess.run([sys.executable, "scripts/extract_dinov2_embeddings.py", "--device", "cuda"], check=True)

## 7. Executar análises e visualizar resultados

A validação repetida usa somente treino+validação e pode levar alguns minutos. O teste oficial permanece separado.

In [ ]:
subprocess.run([sys.executable, "scripts/analyze_dinov2_embeddings.py"], check=True)

In [ ]:
from IPython.display import Image as DisplayImage, display

metrics = pd.read_csv("results/dinov2/tables/linear_probe_metrics.csv")
stability = pd.read_csv("results/dinov2/tables/repeated_probe_summary.csv")
clusters = pd.read_csv("results/dinov2/tables/clustering_metrics.csv")
display(metrics.round(3))
display(stability.round(3))
display(clusters.round(3))
display(DisplayImage(filename="results/dinov2/figures/pca_class_mode.png"))
display(DisplayImage(filename="results/dinov2/figures/prototype_similarity.png"))
display(DisplayImage(filename="results/dinov2/figures/repeated_probe_stability.png"))
display(DisplayImage(filename="results/dinov2/figures/linear_probe_confusions.png"))

## 8. Baixar o pacote de resultados

Guarde o ZIP e envie-o para revisão antes de publicar resultados no GitHub.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    "/content/PetroVision_DINOv2_results",
    "zip",
    root_dir="/content/PetroVision-Multimodal/results",
    base_dir="dinov2",
)
print("Pacote criado:", archive)
files.download(archive)